In [ ]:
pip install pyproj

In [ ]:
#Python

# TWO-WAY LATITUDE/LONGITUDE AND UTM CONVERTER
# Datum: WGS 84

from pyproj import CRS, Transformer


def get_utm_crs(zone, hemisphere):
    """Create the UTM coordinate system for a zone and hemisphere."""
    if not 1 <= zone <= 60:
        raise ValueError("UTM zone must be between 1 and 60.")

    hemisphere = hemisphere.strip().upper()

    if hemisphere not in ("N", "S"):
        raise ValueError("Hemisphere must be N or S.")

    epsg = (32600 if hemisphere == "N" else 32700) + zone
    return CRS.from_epsg(epsg)


def latlon_to_utm(latitude, longitude, zone=None):
    """Convert latitude/longitude to UTM."""
    if not -90 <= latitude <= 90:
        raise ValueError("Latitude must be between -90 and 90.")

    if not -180 <= longitude <= 180:
        raise ValueError("Longitude must be between -180 and 180.")

    # Automatically determine the UTM zone if not supplied
    if zone is None:
        zone = min(60, int((longitude + 180) // 6) + 1)

    hemisphere = "N" if latitude >= 0 else "S"
    utm_crs = get_utm_crs(zone, hemisphere)

    transformer = Transformer.from_crs(
        "EPSG:4326", utm_crs, always_xy=True
    )

    easting, northing = transformer.transform(longitude, latitude)

    return easting, northing, zone, hemisphere


def utm_to_latlon(easting, northing, zone, hemisphere):
    """Convert UTM to latitude/longitude."""
    utm_crs = get_utm_crs(zone, hemisphere)

    transformer = Transformer.from_crs(
        utm_crs, "EPSG:4326", always_xy=True
    )

    longitude, latitude = transformer.transform(easting, northing)

    return latitude, longitude


def utm_to_utm(easting, northing, source_zone, source_hemi,
               target_zone, target_hemi):
    """Convert from one UTM zone/hemisphere to another."""
    latitude, longitude = utm_to_latlon(
        easting, northing, source_zone, source_hemi
    )

    new_e, new_n, _, _ = latlon_to_utm(
        latitude, longitude, target_zone
    )

    # Reproject directly to the requested target hemisphere.
    source_crs = "EPSG:4326"
    target_crs = get_utm_crs(target_zone, target_hemi)

    transformer = Transformer.from_crs(
        source_crs, target_crs, always_xy=True
    )

    new_e, new_n = transformer.transform(longitude, latitude)

    return new_e, new_n, latitude, longitude


def main():
    print("\n===== TWO-WAY UTM CONVERTER =====")
    print("1. Latitude/Longitude to UTM")
    print("2. UTM to Latitude/Longitude")
    print("3. UTM North/South or Zone to another UTM")
    choice = input("Choose 1, 2, or 3: ").strip()

    try:
        if choice == "1":
            lat = float(input("Enter latitude in decimal degrees: "))
            lon = float(input("Enter longitude in decimal degrees: "))

            use_zone = input(
                "Enter UTM zone (press Enter for automatic zone): "
            ).strip()

            zone = int(use_zone) if use_zone else None

            e, n, zone, hemi = latlon_to_utm(lat, lon, zone)

            print("\n--- UTM RESULT ---")
            print(f"Zone: {zone}{hemi}")
            print(f"Easting:  {e:.3f} m")
            print(f"Northing: {n:.3f} m")

        elif choice == "2":
            e = float(input("Enter UTM Easting (m): "))
            n = float(input("Enter UTM Northing (m): "))
            zone = int(input("Enter UTM zone (1-60): "))
            hemi = input("Enter hemisphere (N or S): ").strip().upper()

            lat, lon = utm_to_latlon(e, n, zone, hemi)

            print("\n--- GEOGRAPHIC RESULT ---")
            print(f"Latitude:  {lat:.8f} degrees")
            print(f"Longitude: {lon:.8f} degrees")

        elif choice == "3":
            e = float(input("Enter source UTM Easting (m): "))
            n = float(input("Enter source UTM Northing (m): "))
            sz = int(input("Enter source UTM zone: "))
            sh = input("Enter source hemisphere (N/S): ").strip().upper()

            tz = int(input("Enter target UTM zone: "))
            th = input("Enter target hemisphere (N/S): ").strip().upper()

            new_e, new_n, lat, lon = utm_to_utm(
                e, n, sz, sh, tz, th
            )

            print("\n--- CONVERTED UTM RESULT ---")
            print(f"Target UTM: Zone {tz}{th}")
            print(f"Easting:  {new_e:.3f} m")
            print(f"Northing: {new_n:.3f} m")
            print(f"Latitude:  {lat:.8f} degrees")
            print(f"Longitude: {lon:.8f} degrees")

        else:
            print("Invalid choice. Please choose 1, 2, or 3.")

    except (ValueError, TypeError) as error:
        print("Input error:", error)


if __name__ == "__main__":
    main()


===== TWO-WAY UTM CONVERTER =====
1. Latitude/Longitude to UTM
2. UTM to Latitude/Longitude
3. UTM North/South or Zone to another UTM
Choose 1, 2, or 3: 1
Enter latitude in decimal degrees: 36
Enter longitude in decimal degrees: 30
Enter UTM zone (press Enter for automatic zone): 

--- UTM RESULT ---
Zone: 36N
Easting:  229578.630 m
Northing: 3988111.962 m


In [ ]:
#Python

# TWO-WAY LATITUDE/LONGITUDE AND UTM CONVERTER
# Datum: WGS 84

from pyproj import CRS, Transformer


def get_utm_crs(zone, hemisphere):
    """Create the UTM coordinate system for a zone and hemisphere."""
    if not 1 <= zone <= 60:
        raise ValueError("UTM zone must be between 1 and 60.")

    hemisphere = hemisphere.strip().upper()

    if hemisphere not in ("N", "S"):
        raise ValueError("Hemisphere must be N or S.")

    epsg = (32600 if hemisphere == "N" else 32700) + zone
    return CRS.from_epsg(epsg)


def latlon_to_utm(latitude, longitude, zone=None):
    """Convert latitude/longitude to UTM."""
    if not -90 <= latitude <= 90:
        raise ValueError("Latitude must be between -90 and 90.")

    if not -180 <= longitude <= 180:
        raise ValueError("Longitude must be between -180 and 180.")

    # Automatically determine the UTM zone if not supplied
    if zone is None:
        zone = min(60, int((longitude + 180) // 6) + 1)

    hemisphere = "N" if latitude >= 0 else "S"
    utm_crs = get_utm_crs(zone, hemisphere)

    transformer = Transformer.from_crs(
        "EPSG:4326", utm_crs, always_xy=True
    )

    easting, northing = transformer.transform(longitude, latitude)

    return easting, northing, zone, hemisphere


def utm_to_latlon(easting, northing, zone, hemisphere):
    """Convert UTM to latitude/longitude."""
    utm_crs = get_utm_crs(zone, hemisphere)

    transformer = Transformer.from_crs(
        utm_crs, "EPSG:4326", always_xy=True
    )

    longitude, latitude = transformer.transform(easting, northing)

    return latitude, longitude


def utm_to_utm(easting, northing, source_zone, source_hemi,
               target_zone, target_hemi):
    """Convert from one UTM zone/hemisphere to another."""
    latitude, longitude = utm_to_latlon(
        easting, northing, source_zone, source_hemi
    )

    new_e, new_n, _, _ = latlon_to_utm(
        latitude, longitude, target_zone
    )

    # Reproject directly to the requested target hemisphere.
    source_crs = "EPSG:4326"
    target_crs = get_utm_crs(target_zone, target_hemi)

    transformer = Transformer.from_crs(
        source_crs, target_crs, always_xy=True
    )

    new_e, new_n = transformer.transform(longitude, latitude)

    return new_e, new_n, latitude, longitude


def main():
    print("\n===== TWO-WAY UTM CONVERTER =====")
    print("1. Latitude/Longitude to UTM")
    print("2. UTM to Latitude/Longitude")
    print("3. UTM North/South or Zone to another UTM")
    choice = input("Choose 1, 2, or 3: ").strip()

    try:
        if choice == "1":
            lat = float(input("Enter latitude in decimal degrees: "))
            lon = float(input("Enter longitude in decimal degrees: "))

            use_zone = input(
                "Enter UTM zone (press Enter for automatic zone): "
            ).strip()

            zone = int(use_zone) if use_zone else None

            e, n, zone, hemi = latlon_to_utm(lat, lon, zone)

            print("\n--- UTM RESULT ---")
            print(f"Zone: {zone}{hemi}")
            print(f"Easting:  {e:.3f} m")
            print(f"Northing: {n:.3f} m")

        elif choice == "2":
            e = float(input("Enter UTM Easting (m): "))
            n = float(input("Enter UTM Northing (m): "))
            zone = int(input("Enter UTM zone (1-60): "))
            hemi = input("Enter hemisphere (N or S): ").strip().upper()

            lat, lon = utm_to_latlon(e, n, zone, hemi)

            print("\n--- GEOGRAPHIC RESULT ---")
            print(f"Latitude:  {lat:.8f} degrees")
            print(f"Longitude: {lon:.8f} degrees")

        elif choice == "3":
            e = float(input("Enter source UTM Easting (m): "))
            n = float(input("Enter source UTM Northing (m): "))
            sz = int(input("Enter source UTM zone: "))
            sh = input("Enter source hemisphere (N/S): ").strip().upper()

            tz = int(input("Enter target UTM zone: "))
            th = input("Enter target hemisphere (N/S): ").strip().upper()

            new_e, new_n, lat, lon = utm_to_utm(
                e, n, sz, sh, tz, th
            )

            print("\n--- CONVERTED UTM RESULT ---")
            print(f"Target UTM: Zone {tz}{th}")
            print(f"Easting:  {new_e:.3f} m")
            print(f"Northing: {new_n:.3f} m")
            print(f"Latitude:  {lat:.8f} degrees")
            print(f"Longitude: {lon:.8f} degrees")

        else:
            print("Invalid choice. Please choose 1, 2, or 3.")

    except (ValueError, TypeError) as error:
        print("Input error:", error)


if __name__ == "__main__":
    main()


===== TWO-WAY UTM CONVERTER =====
1. Latitude/Longitude to UTM
2. UTM to Latitude/Longitude
3. UTM North/South or Zone to another UTM
Choose 1, 2, or 3: 2
Enter UTM Easting (m): 229578.630
Enter UTM Northing (m): 3988111.962
Enter UTM zone (1-60): 36
Enter hemisphere (N or S): N

--- GEOGRAPHIC RESULT ---
Latitude:  36.00000000 degrees
Longitude: 30.00000000 degrees


In [ ]:
#Python

# TWO-WAY LATITUDE/LONGITUDE AND UTM CONVERTER
# Datum: WGS 84

from pyproj import CRS, Transformer


def get_utm_crs(zone, hemisphere):
    """Create the UTM coordinate system for a zone and hemisphere."""
    if not 1 <= zone <= 60:
        raise ValueError("UTM zone must be between 1 and 60.")

    hemisphere = hemisphere.strip().upper()

    if hemisphere not in ("N", "S"):
        raise ValueError("Hemisphere must be N or S.")

    epsg = (32600 if hemisphere == "N" else 32700) + zone
    return CRS.from_epsg(epsg)


def latlon_to_utm(latitude, longitude, zone=None):
    """Convert latitude/longitude to UTM."""
    if not -90 <= latitude <= 90:
        raise ValueError("Latitude must be between -90 and 90.")

    if not -180 <= longitude <= 180:
        raise ValueError("Longitude must be between -180 and 180.")

    # Automatically determine the UTM zone if not supplied
    if zone is None:
        zone = min(60, int((longitude + 180) // 6) + 1)

    hemisphere = "N" if latitude >= 0 else "S"
    utm_crs = get_utm_crs(zone, hemisphere)

    transformer = Transformer.from_crs(
        "EPSG:4326", utm_crs, always_xy=True
    )

    easting, northing = transformer.transform(longitude, latitude)

    return easting, northing, zone, hemisphere


def utm_to_latlon(easting, northing, zone, hemisphere):
    """Convert UTM to latitude/longitude."""
    utm_crs = get_utm_crs(zone, hemisphere)

    transformer = Transformer.from_crs(
        utm_crs, "EPSG:4326", always_xy=True
    )

    longitude, latitude = transformer.transform(easting, northing)

    return latitude, longitude


def utm_to_utm(easting, northing, source_zone, source_hemi,
               target_zone, target_hemi):
    """Convert from one UTM zone/hemisphere to another."""
    latitude, longitude = utm_to_latlon(
        easting, northing, source_zone, source_hemi
    )

    new_e, new_n, _, _ = latlon_to_utm(
        latitude, longitude, target_zone
    )

    # Reproject directly to the requested target hemisphere.
    source_crs = "EPSG:4326"
    target_crs = get_utm_crs(target_zone, target_hemi)

    transformer = Transformer.from_crs(
        source_crs, target_crs, always_xy=True
    )

    new_e, new_n = transformer.transform(longitude, latitude)

    return new_e, new_n, latitude, longitude


def main():
    print("\n===== TWO-WAY UTM CONVERTER =====")
    print("1. Latitude/Longitude to UTM")
    print("2. UTM to Latitude/Longitude")
    print("3. UTM North/South or Zone to another UTM")
    choice = input("Choose 1, 2, or 3: ").strip()

    try:
        if choice == "1":
            lat = float(input("Enter latitude in decimal degrees: "))
            lon = float(input("Enter longitude in decimal degrees: "))

            use_zone = input(
                "Enter UTM zone (press Enter for automatic zone): "
            ).strip()

            zone = int(use_zone) if use_zone else None

            e, n, zone, hemi = latlon_to_utm(lat, lon, zone)

            print("\n--- UTM RESULT ---")
            print(f"Zone: {zone}{hemi}")
            print(f"Easting:  {e:.3f} m")
            print(f"Northing: {n:.3f} m")

        elif choice == "2":
            e = float(input("Enter UTM Easting (m): "))
            n = float(input("Enter UTM Northing (m): "))
            zone = int(input("Enter UTM zone (1-60): "))
            hemi = input("Enter hemisphere (N or S): ").strip().upper()

            lat, lon = utm_to_latlon(e, n, zone, hemi)

            print("\n--- GEOGRAPHIC RESULT ---")
            print(f"Latitude:  {lat:.8f} degrees")
            print(f"Longitude: {lon:.8f} degrees")

        elif choice == "3":
            e = float(input("Enter source UTM Easting (m): "))
            n = float(input("Enter source UTM Northing (m): "))
            sz = int(input("Enter source UTM zone: "))
            sh = input("Enter source hemisphere (N/S): ").strip().upper()

            tz = int(input("Enter target UTM zone: "))
            th = input("Enter target hemisphere (N/S): ").strip().upper()

            new_e, new_n, lat, lon = utm_to_utm(
                e, n, sz, sh, tz, th
            )

            print("\n--- CONVERTED UTM RESULT ---")
            print(f"Target UTM: Zone {tz}{th}")
            print(f"Easting:  {new_e:.3f} m")
            print(f"Northing: {new_n:.3f} m")
            print(f"Latitude:  {lat:.8f} degrees")
            print(f"Longitude: {lon:.8f} degrees")

        else:
            print("Invalid choice. Please choose 1, 2, or 3.")

    except (ValueError, TypeError) as error:
        print("Input error:", error)


if __name__ == "__main__":
    main()


===== TWO-WAY UTM CONVERTER =====
1. Latitude/Longitude to UTM
2. UTM to Latitude/Longitude
3. UTM North/South or Zone to another UTM
Choose 1, 2, or 3: 3
Enter source UTM Easting (m): 229578.630
Enter source UTM Northing (m): 3988111.962
Enter source UTM zone: 36
Enter source hemisphere (N/S): N
Enter target UTM zone: 35
Enter target hemisphere (N/S): S

--- CONVERTED UTM RESULT ---
Target UTM: Zone 35S
Easting:  770421.370 m
Northing: 13988111.962 m
Latitude:  36.00000000 degrees
Longitude: 30.00000000 degrees
